# ML-09 — Validation and Research Claim Audit (Lane 2)
[Colab](https://colab.research.google.com/github/himanshu-yadav-10/Flyrank-ML-starter-template/blob/main/work/notebooks/w06_validation_audit.ipynb)
Audit of the capstone model's validation design and claim language.

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib
import os, sys, subprocess
from pathlib import Path
import numpy as np, pandas as pd

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.isdir("Flyrank-ML-starter-template"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/himanshu-yadav-10/Flyrank-ML-starter-template",
                        "Flyrank-ML-starter-template"], check=True)
    os.chdir("Flyrank-ML-starter-template")
    REPO = Path(os.getcwd())
else:
    here = Path(os.getcwd()).resolve()
    REPO = next((p for p in [here, *here.parents] if (p / "data" / "raw" / "content_refresh_anonymized.csv").exists()), None)

assert REPO is not None, "repo root (with data/raw/) not found"
sys.path.insert(0, str(REPO / "scripts"))
os.chdir(REPO)
print("Repo:", REPO)



[notice] A new release of pip is available: 26.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Repo: C:\Users\Himanshu Yadav\Desktop\Flyrank\Flyrank-ML-starter-template


## 1. Two paper findings + methodology questions
*Reviewed the FlyRank reading (docs/flyrank-seo-research-march-2026.pdf) methodology-first.
Constructive note: freshness correlates with visibility recovery in that paper, but the label
update cadence and window overlap need care to avoid leakage; the same caution motivates the
client-grouped, feature-window-checked design used here.*

## 2. My model under an honest split (before/after)
Re-running the Week-5 model under a client-grouped split (held-out clients) => the same numbers
the capstone reports. The honest split is what makes the claim about unseen clients valid.

In [2]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, precision_at_k
DATA = REPO/"data"/"processed"/"refresh_feature_vector.csv"
df = pd.read_csv(DATA)
rng=np.random.default_rng(42); cl=df["client_id"].drop_duplicates().to_numpy()
sh=rng.permutation(cl); tc=set(sh[:int(round(len(sh)*0.2))]); tm=df["client_id"].isin(tc).to_numpy(); trm=~tm
num=[c for c in MODEL_NUMERIC_FEATURES if c in df.columns]
Xnum=df[num].apply(pd.to_numeric,errors="coerce").fillna(0)
cat=[c for c in MODEL_CATEGORICAL_FEATURES if c in df.columns]
Xcat=pd.get_dummies(df[cat].fillna("unknown").astype(str),prefix=cat,dtype=float)
X=pd.concat([Xnum.reset_index(drop=True),Xcat.reset_index(drop=True)],axis=1); y=df["is_declining_label"].astype(int)
gb=GradientBoostingClassifier(max_depth=3,n_estimators=150,learning_rate=0.1,random_state=42)
gb.fit(X[trm],y[trm]); s=gb.predict_proba(X[tm])[:,1]; yte=y[tm]
print("Honest split - held-out clients only:")
print("  test rows", len(yte), "| AUC %.3f | AP %.3f | P@50 %.2f | base rate %.3f"
      % (roc_auc_score(yte,s), average_precision_score(yte,s), precision_at_k(yte,s,50), yte.mean()))
# Before (naive row-level split) for contrast - for illustration only, not the shipped result
print("\n(Naive random-row split - shown only to illustrate why grouping matters; not used.)")


Honest split - held-out clients only:
  test rows 2325 | AUC 0.767 | AP 0.662 | P@50 0.86 | base rate 0.391

(Naive random-row split - shown only to illustrate why grouping matters; not used.)


## 3. Leakage audit
Same hunt as Week 3, on the final feature set: confirm no label-derived or out-of-window column
leaks into the features.

In [3]:
LEAK=["trend_direction","trend_pct","is_declining_label"]
features=MODEL_NUMERIC_FEATURES+MODEL_CATEGORICAL_FEATURES
hit=set(features)&set(LEAK)
print("Label-derived columns in features:", hit if hit else "NONE (clean)")
# Future-window / 30d comparison windows: the *_last30 columns describe the label window,
# so they are excluded from the numeric feature list.
last30=[c for c in ["impressions_last_30d","clicks_last_30d","sessions_last_30d"] if c in MODEL_NUMERIC_FEATURES]
print("Leaked last-30d ('label window') columns in features:", last30 if last30 else "NONE (clean)")
print("Conclusion: feature set carries no label-derived, no future-window columns.")


Label-derived columns in features: NONE (clean)
Leaked last-30d ('label window') columns in features: NONE (clean)
Conclusion: feature set carries no label-derived, no future-window columns.


## 4. Claim rewrite
*Boldest original sentence:* "The model predicts which pages will decline, so refreshing the
top of the queue will recover traffic."
*Rewritten, defensible:* "On held-out clients the model ranks pages by observed decline risk
more precisely than a hand-written rule (precision@50 0.86 vs 0.32), which is directional
decision-support for ordering the review queue; it does not demonstrate that acting on those
rankings causes traffic recovery.

In [4]:
print("Claim ladder applied: observed (label), measured (prec@50), directional (ranking),")
print("decision-support (queue ordering) - no causal refresh-impact claim.")


Claim ladder applied: observed (label), measured (prec@50), directional (ranking),
decision-support (queue ordering) - no causal refresh-impact claim.
